# Semaine 4 — Jour 3 : Tool Registry — Teacher

Source de vérité : fichiers Markdown de `book/week04/day03/`.

# Objectifs d’apprentissage

À la fin de cette journée, l’apprenant saura :

## Objectifs conceptuels

- expliquer pourquoi un agent ne doit pas appeler directement des fonctions métier non contrôlées ;
- distinguer `ToolDefinition`, `ToolRegistry`, `ToolContext` et `ToolResult` ;
- décrire le cycle `discover → validate → authorize → execute → trace` ;
- comprendre les risques liés aux tools sensibles ;
- justifier l’usage d’un schéma d’entrée explicite ;
- séparer description publique d’un outil et handler Python interne.

## Objectifs pratiques

- implémenter un registre d’outils en Python ;
- enregistrer un tool avec son schéma d’entrée ;
- lister les tools disponibles pour un agent ;
- valider des arguments avant appel ;
- appliquer une politique de scope et d’approbation humaine ;
- capturer une exception de handler sans casser l’agent ;
- exporter un manifest JSON sérialisable.

## Objectifs AI Engineering

- concevoir une frontière sûre entre modèle et application ;
- préparer l’intégration future avec un runner, un workflow engine et l’observabilité ;
- produire du code testable sans dépendre d’une clé API ;
- raisonner en termes de contrats, d’invariants et de surfaces d’attaque.


# Chapitre — Concevoir un Tool Registry

## 1. Pourquoi un registre d’outils ?

Un agent sans outil peut analyser, reformuler ou produire du texte. Un agent avec outils peut lire une base de connaissance, créer un ticket, appeler une API, interroger une base SQL ou déclencher un workflow.

Cette puissance crée immédiatement un problème d’architecture : le modèle ne doit pas pouvoir invoquer n’importe quelle fonction avec n’importe quels arguments.

Un `ToolRegistry` répond à ce problème. Il centralise :

| Responsabilité | Description |
|---|---|
| Découverte | Quels outils sont disponibles ? |
| Contrat | Quels arguments sont acceptés ? |
| Validation | Les arguments fournis sont-ils conformes ? |
| Autorisation | L’utilisateur ou l’agent a-t-il le droit d’appeler ce tool ? |
| Exécution | Comment appeler le handler réel ? |
| Résultat | Comment normaliser la sortie ? |
| Trace | Que s’est-il passé pendant l’appel ? |

Le registre d’outils est donc une frontière technique. Il protège le système contre les appels implicites, les arguments invalides, les actions prématurées et les comportements difficiles à déboguer.

## 2. Position dans le mini-framework

```mermaid
flowchart TD
    User[Utilisateur] --> Agent[Agent]
    Agent --> Runner[Runner]
    Runner --> Registry[ToolRegistry]
    Registry --> Policy[Policy checks]
    Registry --> Validator[Schema validator]
    Registry --> Handler[Python handler]
    Handler --> External[API / DB / Service]
    Registry --> Result[ToolResult]
    Result --> Agent
```

L’agent ne connaît pas les détails d’exécution. Il reçoit une liste de tools disponibles et choisit, via le modèle ou via une logique déterministe, un appel structuré.

Le registre prend ensuite le relais.

## 3. Contrat minimal d’un outil

Dans cette journée, un outil possède :

- `name` : identifiant stable ;
- `description` : description orientée modèle ;
- `input_schema` : schéma d’arguments ;
- `output_schema` : forme attendue de la sortie ;
- `sensitive` : indique si l’outil requiert une approbation humaine ;
- `required_scope` : permission applicative minimale ;
- `enabled` : activation ou désactivation ;
- `tags` : classification utile pour filtrage et documentation ;
- `handler` : fonction Python interne.

Le handler ne doit jamais être exposé dans le manifest public. Le modèle voit le contrat, pas le callable Python.

## 4. Cycle d’exécution

```mermaid
sequenceDiagram
    participant A as Agent
    participant R as ToolRegistry
    participant V as Validator
    participant P as Policy
    participant H as Handler
    participant T as Trace

    A->>R: call(tool_name, arguments, context)
    R->>T: registry.lookup
    R->>P: check enabled/scope/approval
    P-->>R: allowed
    R->>V: validate arguments
    V-->>R: normalized arguments
    R->>H: execute(arguments, context)
    H-->>R: output
    R->>T: tool.completed
    R-->>A: ToolResult
```

Le cycle volontairement simple est :

```text
discover → validate → authorize → execute → trace
```

En production, on peut ajouter des timeouts, retries, circuit breakers, quotas, sandboxing, audit logs et redaction automatique. Pour cette journée, on garde un noyau pédagogique mais testable.

## 5. Validation des arguments

Le lab implémente un sous-ensemble de JSON Schema :

- racine `type: object` ;
- `properties` ;
- `required` ;
- `additionalProperties`;
- types simples : `string`, `integer`, `number`, `boolean`, `object`, `array` ;
- `enum` ;
- `default`.

Exemple :

```json
{
  "type": "object",
  "properties": {
    "title": {
      "type": "string",
      "description": "Titre du ticket"
    },
    "priority": {
      "type": "string",
      "enum": ["low", "medium", "high"],
      "default": "medium"
    }
  },
  "required": ["title"],
  "additionalProperties": false
}
```

Cette validation n’est pas un validateur JSON Schema complet. Elle est suffisante pour enseigner l’invariant fondamental : aucun outil n’est exécuté avant validation.

## 6. Autorisation et outils sensibles

Un tool peut être techniquement valide mais métierement dangereux.

Exemples :

| Tool | Risque |
|---|---|
| `send_email` | communication externe non souhaitée |
| `create_ticket` | bruit opérationnel |
| `refund_customer` | impact financier |
| `delete_file` | perte de données |
| `run_shell_command` | risque système |

Le lab introduit deux contrôles :

1. `required_scope` : l’utilisateur ou le workflow doit posséder une permission.
2. `sensitive` : une approbation explicite doit être présente dans le contexte.

Un outil sensible sans approbation retourne `blocked`, pas `failed`. Ce détail est important : le système fonctionne correctement, mais refuse une action non autorisée.

## 7. Résultat normalisé

Chaque appel retourne un `ToolResult` :

```python
ToolResult(
    tool_name="add_numbers",
    status="completed",
    output={"sum": 42},
    error=None,
    elapsed_ms=1,
    trace=(...)
)
```

Les statuts utilisés dans le lab sont :

| Statut | Sens |
|---|---|
| `completed` | l’outil a terminé correctement |
| `blocked` | une politique empêche l’appel |
| `failed` | erreur de schéma, outil inconnu ou exception handler |

Ce contrat prépare l’intégration avec le runner du framework. Le runner n’aura pas besoin de connaître les exceptions internes du tool.

## 8. Décorateur `@tool`

Le lab propose aussi un décorateur pédagogique :

```python
@tool(description="Multiplie deux entiers.")
def multiply(a: int, b: int) -> dict:
    return {"product": a * b}
```

Le décorateur extrait les annotations simples et génère un schéma d’entrée. Cette mécanique imite le confort développeur attendu dans un framework, tout en gardant le contrat explicite.

## 9. Erreurs à éviter

### Exposer tous les tools à tous les agents

Un agent support client n’a pas besoin d’un outil de suppression de données. La liste des tools doit être adaptée au rôle, au contexte et aux permissions.

### Confondre validation modèle et validation serveur

Même si un modèle produit des arguments structurés, le serveur doit revalider. Le modèle propose ; le runtime contrôle.

### Laisser remonter les exceptions brutes

Une exception Python brute ne doit pas casser toute la boucle agentique. Elle doit être capturée et convertie en résultat exploitable.

### Mélanger logs, traces et réponse utilisateur

La trace est destinée au debug et à l’observabilité. Elle ne doit pas être automatiquement exposée à l’utilisateur final.

## 10. Préparation du jour suivant

Le jour 4 ajoutera la `Memory Layer`. Le registre d’outils doit donc rester indépendant de la mémoire. Un outil peut recevoir un contexte, mais il ne doit pas devenir lui-même un store d’état global.

## Propositions d’amélioration

Ces pistes ne modifient pas la spécification figée du projet :

- ajouter un timeout par outil ;
- connecter le registre à une couche d’observabilité structurée ;
- introduire un validateur JSON Schema complet ;
- ajouter un système de quotas par utilisateur ;
- générer automatiquement une documentation Markdown des tools.


# Exercices

## Exercice 1 — Identifier les responsabilités

Pour chaque responsabilité ci-dessous, indique si elle appartient plutôt à l’agent, au registre d’outils ou au handler métier.

| Responsabilité | Composant attendu |
|---|---|
| Choisir qu’un outil peut être utile | |
| Valider que `priority` vaut `low`, `medium` ou `high` | |
| Créer réellement un ticket dans un système support | |
| Vérifier qu’un utilisateur possède le scope `ticket:write` | |
| Transformer une exception en résultat normalisé | |
| Construire la réponse finale à l’utilisateur | |

## Exercice 2 — Concevoir un schéma

Écris un schéma d’entrée pour un outil `lookup_customer`.

Contraintes :

- `customer_id` est obligatoire ;
- `customer_id` est une chaîne ;
- `include_orders` est un booléen optionnel avec valeur par défaut `false` ;
- aucun champ inattendu n’est autorisé.

## Exercice 3 — Ajouter un outil non sensible

Dans le lab, ajoute un outil `calculate_discount`.

Contraintes :

- arguments : `price` nombre, `percentage` nombre ;
- les deux champs sont obligatoires ;
- l’outil retourne `{"discounted_price": ...}` ;
- l’outil n’est pas sensible.

## Exercice 4 — Ajouter une politique

Modifie l’outil `calculate_discount` pour exiger le scope `pricing:read`.

Teste deux contextes :

1. sans scope ;
2. avec scope.

## Exercice 5 — Raisonnement d’architecture

Explique pourquoi le modèle ne doit jamais appeler directement une fonction Python métier sans passer par un registre.


# Questions d’entretien

## Question 1

Quel est le rôle principal d’un `ToolRegistry` dans un framework d’agents ?

## Question 2

Pourquoi faut-il valider les arguments d’un tool côté runtime même si le modèle produit une sortie structurée ?

## Question 3

Quelle différence fais-tu entre un tool `failed` et un tool `blocked` ?

## Question 4

Comment exposerais-tu un outil sensible comme `refund_customer` à un agent en production ?

## Question 5

Pourquoi est-il dangereux d’exposer la totalité des outils disponibles à tous les agents ?

## Question 6

Quelles informations dois-tu mettre dans une trace d’appel d’outil ?


# Challenge — Registry de support client contrôlé

## Contexte

Tu construis un assistant support interne pour une entreprise SaaS. L’agent doit pouvoir rechercher des politiques, classifier une demande et créer un ticket si nécessaire.

## Objectif

Étendre le lab pour créer un registre d’outils de support client.

## Outils à implémenter

### `classify_issue`

Arguments :

- `message` : string obligatoire.

Sortie attendue :

```json
{
  "category": "billing|technical|security|other",
  "confidence": 0.0
}
```

### `search_policy`

Réutilise ou adapte l’outil existant.

### `create_ticket`

Arguments :

- `title` : string obligatoire ;
- `category` : string obligatoire, enum `billing`, `technical`, `security`, `other` ;
- `priority` : string optionnel, enum `low`, `medium`, `high`.

Contraintes :

- outil sensible ;
- scope requis : `ticket:write` ;
- approbation humaine obligatoire.

## Critères d’acceptation

- le manifest n’expose pas les handlers Python ;
- `create_ticket` est masqué par défaut ;
- un appel sans scope est bloqué ;
- un appel sans approbation est bloqué ;
- un appel avec scope et approbation réussit ;
- les arguments invalides retournent un statut `failed` ;
- chaque appel produit une trace lisible.

## Variante avancée

Ajoute une méthode `list_tools_for_agent(agent_role)` qui filtre les outils selon le rôle :

- `support_reader` : lecture seulement ;
- `support_operator` : lecture + création de ticket avec approbation ;
- `admin` : tous les outils.


# Références

## Références principales

- OpenAI Agents SDK — Tools : `https://openai.github.io/openai-agents-python/tools/`
- OpenAI Agents SDK — Agents : `https://openai.github.io/openai-agents-python/agents/`
- OpenAI API — Function calling / tools : `https://platform.openai.com/docs/guides/function-calling`
- JSON Schema — Understanding JSON Schema : `https://json-schema.org/understanding-json-schema/`

## Références internes

- Semaine 2 — Jour 2 : Function Calling
- Semaine 2 — Jour 3 : Structured Outputs
- Semaine 4 — Jour 1 : Architecture
- Semaine 4 — Jour 2 : Abstraction Agent

## Points à retenir

- Un tool est une capacité applicative, pas seulement une fonction.
- Le modèle propose un appel ; le runtime valide et autorise.
- Le registre d’outils est une surface de sécurité.
- Les traces d’outils préparent l’observabilité du jour 6.


## Lab exécutable

Le code ci-dessous montre comment utiliser le registre d’outils depuis le notebook.

In [ ]:
import sys
from pathlib import Path

def find_project_root(start: Path) -> Path:
    for parent in [start, *start.parents]:
        if (parent / "mini_framework").exists():
            return parent
    raise RuntimeError("Racine projet introuvable")

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from mini_framework.tool_registry import ToolContext, build_demo_registry

registry = build_demo_registry()
context = ToolContext(user_id="learner", session_id="notebook", scopes=frozenset({"policy:read"}))
result = registry.call("add_numbers", {"a": 20, "b": 22}, context)
result.to_dict()

In [ ]:
# À vous :
# 1. Listez les tools visibles par défaut.
# 2. Appelez search_policy avec et sans scope.
# 3. Observez la différence entre blocked et failed.
registry.list_tools()

# Corrections formateur

# Corrigé — Exercices

## Exercice 1

| Responsabilité | Composant attendu |
|---|---|
| Choisir qu’un outil peut être utile | Agent ou modèle via le runner |
| Valider que `priority` vaut `low`, `medium` ou `high` | ToolRegistry |
| Créer réellement un ticket dans un système support | Handler métier |
| Vérifier qu’un utilisateur possède le scope `ticket:write` | ToolRegistry |
| Transformer une exception en résultat normalisé | ToolRegistry |
| Construire la réponse finale à l’utilisateur | Agent |

## Exercice 2

```json
{
  "type": "object",
  "properties": {
    "customer_id": {
      "type": "string",
      "description": "Identifiant stable du client"
    },
    "include_orders": {
      "type": "boolean",
      "description": "Inclure ou non les commandes associées",
      "default": false
    }
  },
  "required": ["customer_id"],
  "additionalProperties": false
}
```

## Exercice 3

Exemple d’enregistrement :

```python
def calculate_discount(arguments, context):
    price = arguments["price"]
    percentage = arguments["percentage"]
    return {"discounted_price": price * (1 - percentage / 100)}

registry.register(
    "calculate_discount",
    "Calcule un prix remisé.",
    {
        "type": "object",
        "properties": {
            "price": {"type": "number"},
            "percentage": {"type": "number"}
        },
        "required": ["price", "percentage"],
        "additionalProperties": False
    },
    calculate_discount,
)
```

## Exercice 4

Modification :

```python
registry.register(
    "calculate_discount",
    "Calcule un prix remisé.",
    schema,
    calculate_discount,
    required_scope="pricing:read",
)
```

Résultat attendu :

- sans `pricing:read` : `blocked` ;
- avec `pricing:read` : `completed`.

## Exercice 5

Le modèle ne doit pas appeler directement une fonction métier parce que cela supprime la frontière de contrôle. Sans registre, il devient difficile de valider les arguments, filtrer les outils, vérifier les permissions, bloquer les actions sensibles, tracer les appels et transformer les erreurs en résultats exploitables.


# Corrigé — Questions d’entretien

## Question 1

Un `ToolRegistry` centralise la découverte, la description, la validation, l’autorisation, l’exécution et la traçabilité des tools disponibles pour un agent. Il sert de frontière entre le raisonnement du modèle et les capacités applicatives réelles.

## Question 2

La validation runtime reste obligatoire parce qu’une sortie structurée générée par un modèle n’est pas une garantie de sécurité. Le serveur doit protéger ses invariants même si le modèle se trompe, hallucine un champ, oublie un champ requis ou produit une valeur hors enum.

## Question 3

`failed` signifie que l’appel a échoué : outil inconnu, arguments invalides ou exception pendant l’exécution. `blocked` signifie que le système a volontairement refusé l’appel pour une raison de politique : outil désactivé, scope manquant ou approbation humaine absente.

## Question 4

Un outil comme `refund_customer` doit être marqué sensible, limité par scope, journalisé, soumis à approbation humaine, testé avec des montants limites et entouré de garde-fous métier. Le modèle ne doit jamais déclencher directement le remboursement final sans contrôle applicatif.

## Question 5

Exposer tous les outils augmente la surface d’attaque et le risque d’action non pertinente. Un agent doit recevoir seulement les capacités nécessaires à son rôle et à la tâche courante.

## Question 6

Une trace utile contient au minimum le nom du tool, l’instant de lookup, le résultat de validation, les décisions de politique, le statut final, le temps d’exécution, les erreurs normalisées et un identifiant de session ou de run.


# Corrigé — Challenge

## Implémentation indicative

```python
def classify_issue(arguments, context):
    message = arguments["message"].lower()
    if "invoice" in message or "billing" in message:
        return {"category": "billing", "confidence": 0.82}
    if "password" in message or "breach" in message:
        return {"category": "security", "confidence": 0.86}
    if "error" in message or "bug" in message:
        return {"category": "technical", "confidence": 0.78}
    return {"category": "other", "confidence": 0.55}
```

Schéma :

```json
{
  "type": "object",
  "properties": {
    "message": {
      "type": "string"
    }
  },
  "required": ["message"],
  "additionalProperties": false
}
```

Pour `create_ticket`, le point important est moins le code métier que la politique :

```python
registry.register(
    "create_ticket",
    "Crée un ticket support après validation humaine.",
    create_ticket_schema,
    create_ticket,
    sensitive=True,
    required_scope="ticket:write",
    tags=["support", "sensitive"],
)
```

## Tests attendus

- `registry.list_tools()` ne contient pas `create_ticket` ;
- `registry.list_tools(include_sensitive=True)` contient `create_ticket` ;
- contexte sans scope : résultat `blocked` ;
- contexte avec scope mais sans approbation : résultat `blocked` ;
- contexte avec scope et approbation : résultat `completed` ;
- catégorie hors enum : résultat `failed` ;
- chaque résultat contient `trace`.

## Exemple de filtre par rôle

```python
ROLE_TAGS = {
    "support_reader": {"knowledge"},
    "support_operator": {"knowledge", "support"},
    "admin": {"knowledge", "support", "sensitive"},
}

def list_tools_for_agent(registry, role):
    allowed_tags = ROLE_TAGS[role]
    return [
        tool
        for tool in registry.list_tools(include_sensitive=(role == "admin"))
        if set(tool["tags"]) & allowed_tags
    ]
```

Cette solution reste volontairement simple. En production, les rôles seraient probablement gérés par une politique centralisée.


# Review formateur — S4 J3

## Intention pédagogique

Cette journée transforme la notion de tool en composant de framework. Les apprenants doivent sortir du simple “function calling” et raisonner comme des ingénieurs backend : contrats, permissions, erreurs, traces et frontières d’exécution.

## Points à vérifier

- L’apprenant distingue bien tool public et handler interne.
- Il comprend que la validation serveur reste obligatoire.
- Il n’expose pas les outils sensibles par défaut.
- Il utilise `blocked` pour une politique refusée et `failed` pour une erreur.
- Il produit des tests sur les cas négatifs, pas seulement sur les cas heureux.
- Il ne mélange pas tool registry et memory layer.

## Démonstration recommandée

1. Lister les tools sans `include_sensitive`.
2. Appeler `add_numbers` correctement.
3. Appeler `add_numbers` avec un champ extra.
4. Appeler `search_policy` sans scope.
5. Appeler `create_ticket` avec scope mais sans approbation.
6. Rejouer `create_ticket` avec scope et approbation.
7. Lire la trace produite.

## Questions de relance

- Que se passe-t-il si un modèle hallucine un nom de tool ?
- Où placerais-tu les retries ?
- Comment limites-tu l’exposition des tools selon le rôle de l’agent ?
- Quelles traces seraient nécessaires en production ?
- Comment éviter qu’un outil devienne un accès non contrôlé à la base de données ?

## Critères de validation

Une solution est acceptable si elle :

- possède un registre central ;
- valide les arguments avant exécution ;
- protège les outils sensibles ;
- retourne des résultats normalisés ;
- capture les erreurs handler ;
- reste testable sans fournisseur externe.
